# Chip Picker

Give it a color (`'red'`, `'green'`, `'black'`) and the arm picks that chip out of the blue tape **my chips** box, then can drop it on the bet square.

How it works:
1. Arm goes to the `mychips` pose and takes a picture.
2. Finds the blue tape box. Its 4 corners are at known table spots, so they give a **homography** (pixel -> table x, y).
3. Finds chips of the chosen color inside the box (HSV color threshold), converts the center to table x, y.
4. **IK** (built on `fk_Dofbot` from Part 1) turns x, y, z into joint angles.
5. Hover -> down -> close gripper -> lift -> (bet square) -> down -> open -> lift -> home.

Units: meters, same base frame as `fk_Dofbot` (+y = forward at base 90).

**Run order the first time:** setup -> settings -> helpers -> color test -> calibrate box -> find chips (no motion) -> IK check -> hover test -> gripper test -> pick.

In [ ]:
%run ../Part1/DofbotKinematics.ipynb

import json
import os
import time
import cv2
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)

## Settings (edit these)

In [ ]:
# --- Table layout, in the FK base frame (meters). +y = forward at base 90 ---
# Corners of the blue tape "my chips" box (outer edge of the tape)
BOX_CORNERS = {'A': (0.05, 0.15), 'B': (0.18, 0.15), 'C': (0.18, 0.225), 'D': (0.05, 0.225)}

CHIP_DIAMETER = 0.04     # 4 cm
CHIP_HEIGHT = 0.017      # 1.7 cm

# Where to drop chips on the bet square (x, y) in meters. MEASURE THIS with a ruler.
BET_XY = None            # e.g. (-0.10, 0.18)

# --- Motion heights for the gripper tip (meters above the table) ---
HOVER_Z = 0.06           # travel height above chips
GRAB_Z = 0.006           # fingertips this low when closing on a chip. Raise if it hits the table
PLACE_Z = 0.025          # height to let go over the bet square

# --- Gripper (servo 6). Find these with camera_pose_tester / the gripper test cell below ---
GRIP_OPEN = 40           # wide enough for a 4 cm chip
GRIP_CLOSED = 135        # holding a chip firmly

# --- Colors in HSV (OpenCV: H 0-180, S 0-255, V 0-255). Tune with the color test cell ---
COLORS = {
    'blue':  [((95, 80, 50), (130, 255, 255))],                              # tape
    'red':   [((0, 100, 70), (10, 255, 255)), ((170, 100, 70), (180, 255, 255))],  # red wraps around H=0
    'green': [((40, 70, 50), (85, 255, 255))],
    'black': [((0, 0, 0), (180, 255, 60))],
}

CALIB_FILE = 'chip_box_calib.json'

## Vision helpers

In [ ]:
def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def color_mask(frame, color):
    hsv = cv2.cvtColor(cv2.GaussianBlur(frame, (5, 5), 0), cv2.COLOR_BGR2HSV)
    mask = np.zeros(hsv.shape[:2], np.uint8)
    for lo, hi in COLORS[color]:
        mask |= cv2.inRange(hsv, np.array(lo), np.array(hi))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    return mask


def order_corners(pts):
    """Sort 4 image points clockwise starting at the top-left (same order every time from a fixed pose)."""
    pts = np.array(pts, np.float32).reshape(4, 2)
    s, d = pts.sum(1), pts[:, 1] - pts[:, 0]
    return np.array([pts[s.argmin()], pts[d.argmin()], pts[s.argmax()], pts[d.argmax()]], np.float32)


def find_box(frame):
    """Finds the blue tape box. Returns its 4 outer corners in pixels (ordered), or None."""
    mask = color_mask(frame, 'blue')
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest) < 2000:
        return None
    hull = cv2.convexHull(biggest)
    approx = cv2.approxPolyDP(hull, 0.02 * cv2.arcLength(hull, True), True)
    if len(approx) != 4:
        approx = cv2.boxPoints(cv2.minAreaRect(biggest))   # fallback: best-fit rectangle
    return order_corners(approx)


def box_homography(corners_px, labels):
    """Pixel -> table (meters) using the box corners. labels = which world corner each pixel corner is."""
    world = np.array([BOX_CORNERS[l] for l in labels], np.float32)
    return cv2.getPerspectiveTransform(corners_px, world)


def px_to_table(H, pts):
    pts = np.array(pts, np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H).reshape(-1, 2)


def find_chips(frame, color, H, corners_px, cam_xyz=None):
    """Chips of one color inside the box. Returns a list of dicts sorted closest-to-robot first."""
    # Only look inside the box, a bit in from the tape
    inside = np.zeros(frame.shape[:2], np.uint8)
    cv2.fillPoly(inside, [corners_px.astype(np.int32)], 255)
    inside = cv2.erode(inside, np.ones((15, 15), np.uint8))
    mask = color_mask(frame, color) & inside

    chips = []
    for c in cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
        if cv2.contourArea(c) < 150:
            continue
        # Size check in real units: a chip top is ~12.6 cm^2 (sides add a bit when seen at an angle)
        area_cm2 = cv2.contourArea(px_to_table(H, c[:, 0, :]).astype(np.float32)) * 1e4
        if not 4 <= area_cm2 <= 45:
            continue
        m = cv2.moments(c)
        px = (m['m10'] / m['m00'], m['m01'] / m['m00'])
        x, y = px_to_table(H, [px])[0]
        if cam_xyz is not None:
            # The camera sees the chip's TOP (1.7 cm up), not the table. Pull the point toward the camera to fix that.
            cx, cy, cz = cam_xyz
            k = (cz - CHIP_HEIGHT) / cz
            x, y = cx + (x - cx) * k, cy + (y - cy) * k
        chips.append({'xy': (float(x), float(y)), 'px': px, 'area_cm2': area_cm2, 'contour': c})
    chips.sort(key=lambda ch: np.hypot(*ch['xy']))   # closest to the base = easiest to reach
    return chips


def show(frame, corners_px=None, chips=(), title=''):
    img = frame.copy()
    if corners_px is not None:
        cv2.polylines(img, [corners_px.astype(np.int32)], True, (255, 0, 0), 2)
        for i, (u, v) in enumerate(corners_px):
            cv2.putText(img, str(i), (int(u) + 5, int(v) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    for i, ch in enumerate(chips):
        cv2.drawContours(img, [ch['contour']], -1, (0, 255, 0), 2)
        u, v = map(int, ch['px'])
        cv2.circle(img, (u, v), 4, (0, 0, 255), -1)
        cv2.putText(img, f"#{i} ({ch['xy'][0]*100:.1f},{ch['xy'][1]*100:.1f})cm", (u + 6, v),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## IK helpers

In [ ]:
DOWN = np.array([0, 0, -1.0])
Q_LO, Q_HI = np.zeros(4), np.full(4, 180.0)


def ik_residual(q4, target, w_down):
    Rot, Pot = fk_Dofbot(list(q4) + [90])
    pointing = -Rot.as_matrix()[:, 0]          # the gripper points along -x of the tool frame
    return np.r_[Pot - target, w_down * (pointing - DOWN)]


def ik_Dofbot(target, w_down=0.003, iters=100):
    """Joint angles [q1..q5] (deg) that put the gripper tip at target (x, y, z in meters).
    Position comes first; among good answers it prefers pointing the gripper straight down.
    Same idea as the Jacobian in sensitivity.py: nudge each joint, see how the tip moves, step toward the target."""
    target = np.asarray(target, float)
    base = np.degrees(np.arctan2(target[1], target[0]))
    seeds = [[base, a, b, c] for a in (10, 40, 80) for b in (60, 100) for c in (0, 40)]
    best = None
    for seed in seeds:
        q = np.clip(np.array(seed, float), Q_LO, Q_HI)
        for _ in range(iters):
            r = ik_residual(q, target, w_down)
            J = np.zeros((len(r), 4))
            for i in range(4):
                dq = np.zeros(4)
                dq[i] = 0.01
                J[:, i] = (ik_residual(q + dq, target, w_down) - r) / 0.01
            step = np.clip(-np.linalg.solve(J.T @ J + 1e-6 * np.eye(4), J.T @ r), -10, 10)
            q = np.clip(q + step, Q_LO, Q_HI)
            if np.linalg.norm(step) < 1e-3:
                break
        Rot, Pot = fk_Dofbot(list(q) + [90])
        err = np.linalg.norm(Pot - target)
        pitch = pitch_from_vertical(Rot)
        score = err + 0.00005 * (180 - pitch)       # 1 mm of error ~ 20 deg less tilt
        if best is None or score < best[3]:
            best = (q, err, pitch, score)
    q, err, pitch, _ = best
    return [float(round(a, 1)) for a in q] + [90.0], err, pitch

## Motion helpers

In [ ]:
last_q = [90, 90, 90, 90, 90]   # last joint angles we sent (used to open/close without moving)


def move(q, grip, ms=1500):
    # q = [q1..q5] in degrees, grip = servo 6
    global last_q
    last_q = list(q[:5])
    arm.Arm_serial_servo_write6(*q[:5], grip, ms)
    time.sleep(ms / 1000 + 0.5)


def move_to_xyz(xyz, grip, ms=1500, max_err=0.005):
    q, err, pitch = ik_Dofbot(xyz)
    print(f"  -> ({xyz[0]*100:.1f}, {xyz[1]*100:.1f}, {xyz[2]*100:.1f}) cm  q={q}  err={err*1000:.1f} mm  pitch={pitch:.0f}")
    if err > max_err:
        raise RuntimeError(f"Can't reach that spot (off by {err*1000:.0f} mm). Pick a chip closer to the robot.")
    move(q, grip, ms)
    return q


def look_for_chips(color, show_it=True):
    """Go to the mychips pose, find the box and chips of one color."""
    move(poses['mychips'], GRIP_OPEN)
    frame = take_picture()
    if frame is None:
        raise RuntimeError("No camera frame")
    corners = find_box(frame)
    if corners is None:
        show(frame, title='Blue box not found')
        raise RuntimeError("Can't find the blue tape box. Check the 'blue' color range.")
    H = box_homography(corners, calib['labels'])
    _, cam = fk_Dofbot(poses['mychips'])          # camera ~ at the gripper, used for the height fix
    chips = find_chips(frame, color, H, corners, cam_xyz=cam)
    if show_it:
        show(frame, corners, chips, f"{color}: {len(chips)} chip(s)")
    return chips


def pick_chip(color):
    chips = look_for_chips(color)
    if not chips:
        print(f"No {color} chips in the box")
        return False
    x, y = chips[0]['xy']
    print(f"Picking {color} chip at ({x*100:.1f}, {y*100:.1f}) cm")
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN)            # above the chip
    move_to_xyz((x, y, GRAB_Z), GRIP_OPEN, ms=1000)    # down around it
    move(last_q, GRIP_CLOSED, ms=800)             # close
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, ms=1000) # lift
    return True


def place_on_bet():
    if BET_XY is None:
        raise RuntimeError("Set BET_XY (bet square center in meters) in the settings cell first.")
    x, y = BET_XY
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED)
    move_to_xyz((x, y, PLACE_Z), GRIP_CLOSED, ms=1000)
    move(last_q, GRIP_OPEN, ms=800)               # let go
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN, ms=1000)
    move(poses['home'], GRIP_OPEN)

## 1. Color test (no arm motion besides going to `mychips`)
Shows what each color mask picks up. White = detected. If a color misses chips or grabs the table, tune its range in `COLORS`.
Black is the hardest (shadows look black too): keep its V max low.

In [ ]:
move(poses['mychips'], GRIP_OPEN)
frame = take_picture()
plt.figure(figsize=(15, 9))
plt.subplot(2, 3, 1); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.title('camera'); plt.axis('off')
for i, color in enumerate(COLORS):
    plt.subplot(2, 3, i + 2); plt.imshow(color_mask(frame, color), cmap='gray'); plt.title(color); plt.axis('off')
plt.show()

# Hover over a spot to read its HSV: print the HSV at the image center
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
h, w = hsv.shape[:2]
print("HSV at image center:", hsv[h // 2, w // 2])

## 2. Calibrate the box corners (once, redo if the camera or table moves)
The corners are numbered 0-3 on the picture. Tell it which real corner each one is:
- **A** = (5, 15) cm  **B** = (18, 15) cm  **C** = (18, 22.5) cm  **D** = (5, 22.5) cm

(A/B are the side closer to the robot, A/D the side closer to x = 0.) Saved to `chip_box_calib.json`.

In [ ]:
move(poses['mychips'], GRIP_OPEN)
frame = take_picture()
corners = find_box(frame)
show(frame, corners, title='Box corners')
print("Corner pixels:", corners.round(0).tolist() if corners is not None else None)

labels = [input(f"Corner {i} at pixel {corners[i].round(0).tolist()} is which (A/B/C/D)? ").strip().upper() for i in range(4)]
assert sorted(labels) == ['A', 'B', 'C', 'D'], "Use each of A, B, C, D once"
calib = {'labels': labels}
with open(CALIB_FILE, 'w') as f:
    json.dump(calib, f)
print("Saved", calib)

Next time you can just load it:

In [ ]:
with open(CALIB_FILE) as f:
    calib = json.load(f)
print(calib)

## 3. Find chips (no grabbing)
Check the green outlines and the cm coordinates. Put a ruler down to check one chip: it should be within ~1 cm.

In [ ]:
for color in ['red', 'green', 'black']:
    chips = look_for_chips(color)
    for i, ch in enumerate(chips):
        print(f"  {color} #{i}: x={ch['xy'][0]*100:.1f} cm  y={ch['xy'][1]*100:.1f} cm  (area {ch['area_cm2']:.1f} cm^2)")

## 4. IK check (no motion)
Solves IK for points in the box and checks them with FK. `err` should be ~0 mm. The far corner of the box is out of reach, but chip centers never get that far (they sit 2 cm inside).

In [ ]:
for x, y in [(0.07, 0.17), (0.115, 0.1875), (0.16, 0.205), (0.07, 0.205), (0.16, 0.17)]:
    for z in (HOVER_Z, GRAB_Z):
        q, err, pitch = ik_Dofbot((x, y, z))
        print(f"({x*100:.1f}, {y*100:.1f}, {z*100:.1f}) cm -> q={q}  err={err*1000:.1f} mm  pitch={pitch:.0f}")

## 5. Hover test (safe: stays 6 cm up)
Moves above the first chip of a color, without going down. The gripper should be right over the chip.

In [ ]:
COLOR = 'red'
chips = look_for_chips(COLOR)
if chips:
    x, y = chips[0]['xy']
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN)

## 6. Gripper test
Finds `GRIP_OPEN` / `GRIP_CLOSED`. Hold a chip between the fingers and try values. Put the good ones in the settings cell.

In [ ]:
for g in [GRIP_OPEN, GRIP_CLOSED, GRIP_OPEN]:
    print("gripper ->", g)
    arm.Arm_serial_servo_write(6, g, 800)
    time.sleep(1.5)

## 7. Pick a chip!

In [ ]:
COLOR = 'red'   # 'red', 'green' or 'black'
picked = pick_chip(COLOR)

## 8. Drop it on the bet square
Set `BET_XY` in the settings cell first.

In [ ]:
if picked:
    place_on_bet()

## Done

In [ ]:
move(poses['home'], GRIP_OPEN)
cap.release()